# vLLM Architecture: Continuous Batching and Serving at Scale

The full code of [vLLM Architecture: Continuous Batching and Serving at Scale](https://residual-stream.lvtan71.workers.dev/posts/vllm-architecture-explained/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/vllm-architecture-explained) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/vllm-architecture-explained.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 3 min end to end on the RTX 3090 machine that saved the outputs below.
The step timings are the M4's. On a CUDA GPU a model this small is launch-bound: on an RTX 3090 a decode step took 21 to 24 ms at every batch size, so the bandwidth analysis does not carry over.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 2 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "transformers==5.19.*" huggingface_hub numpy

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "vllm-architecture-explained"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/vllm-architecture-explained


## `engine.py`

The scheduler simulation: static batching, prefill-first and V1-style chunked continuous batching, with a block pool and preemption, on ShareGPT request lengths.

**Note:** Reads the post's cost.npy.

In [2]:
!{sys.executable} engine.py

                          answer tok/s  TTFT p50  TTFT p99  time/token  p99 gap  longest gap
0.5 requests a second


  static, 16                     64   1058.4 s  2302.7 s      69 ms   0.08 s     0.08 s


  static, 128                    91     86.5 s   534.0 s     163 ms   0.28 s     0.28 s


  prefill-first                 101      0.2 s     1.0 s      63 ms   0.31 s     2.41 s


  chunked, 2048                 101      0.2 s     1.0 s      61 ms   0.29 s     1.30 s


...

  chunked, 512                  291      0.4 s     2.1 s     177 ms   0.52 s     0.55 s
chunked, 512, 1.0 request a second, smaller KV pools


  4096 blocks (65,536 tokens):  199 tok/s, TTFT p99    1.4 s, longest gap  0.52 s,    0 preemptions


   512 blocks ( 8,192 tokens):  181 tok/s, TTFT p99  199.9 s, longest gap 28.62 s, 2004 preemptions


   256 blocks ( 4,096 tokens):  129 tok/s, TTFT p99 1114.8 s, longest gap 28.15 s, 2225 preemptions


   128 blocks ( 2,048 tokens):   79 tok/s, TTFT p99 3085.6 s, longest gap 42.55 s, 2190 preemptions


The post printed:

```text
                          answer tok/s  TTFT p50  TTFT p99  time/token  p99 gap  longest gap
0.5 requests a second
  static, 16                     64   1058.4 s  2302.7 s      69 ms   0.08 s     0.08 s
  static, 128                    91     86.5 s   534.0 s     163 ms   0.28 s     0.28 s
  prefill-first                 101      0.2 s     1.0 s      63 ms   0.31 s     2.41 s
  chunked, 2048                 101      0.2 s     1.0 s      61 ms   0.29 s     1.30 s
  chunked, 512                  101      0.2 s     1.1 s      61 ms   0.31 s     0.50 s
1.0 requests a second
  static, 16                     65   1963.9 s  4189.7 s      69 ms   0.08 s     0.08 s
  static, 128                    98    740.3 s  2099.2 s     232 ms   0.28 s     0.28 s
  prefill-first                 199      0.2 s     1.3 s      99 ms   0.71 s     2.62 s
  chunked, 2048                 199      0.3 s     1.3 s      91 ms   0.58 s     1.55 s
  chunked, 512                  199      0.3 s     1.4 s      91 ms   0.48 s     0.52 s
1.5 requests a second
  static, 16                     66   2280.7 s  4800.7 s      69 ms   0.08 s     0.08 s
  static, 128                    99   1051.2 s  2689.6 s     235 ms   0.28 s     0.28 s
  prefill-first                 289      0.3 s     1.6 s     219 ms   1.45 s     4.19 s
  chunked, 2048                 291      0.4 s     1.9 s     177 ms   0.80 s     1.80 s
  chunked, 512                  291      0.4 s     2.1 s     177 ms   0.52 s     0.55 s
chunked, 512, 1.0 request a second, smaller KV pools
  4096 blocks (65,536 tokens):  199 tok/s, TTFT p99    1.4 s, longest gap  0.52 s,    0 preemptions
   512 blocks ( 8,192 tokens):  181 tok/s, TTFT p99  199.9 s, longest gap 28.62 s, 2004 preemptions
   256 blocks ( 4,096 tokens):  129 tok/s, TTFT p99 1114.8 s, longest gap 28.15 s, 2225 preemptions
   128 blocks ( 2,048 tokens):   79 tok/s, TTFT p99 3085.6 s, longest gap 42.55 s, 2190 preemptions
```

## `lora.py`

The cost of serving many LoRA adapters from one model: shared, gathered, and mixed.

**Note:** GPU.

In [3]:
!{sys.executable} lora.py

one rank-16 adapter on all 7 projections of 28 layers: 35 MiB, against 2.88 GiB for the base model; 1.4% more multiply-adds


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (181013 > 131072). Running this sequence through the model will result in indexing errors
decode step, 512 cached tokens per sequence, ms (extra over the base model)
     base    one adapter, shared   gathered, one adapter   gathered, a different adapter each


 16   21.9      34.8 (+59%)          46.5 (+112%)             46.6 (+113%)


 64   24.3      37.9 (+56%)          47.2 (+94%)             49.5 (+104%)


The post printed:

```text
one rank-16 adapter on all 7 projections of 28 layers: 35 MiB, against 2.88 GiB for the base model; 1.4% more multiply-adds
decode step, 512 cached tokens per sequence, ms (extra over the base model)
     base    one adapter, shared   gathered, one adapter   gathered, a different adapter each
 16   87.0      90.1 ( +4%)         149.5 (+72%)            150.1 (+72%)
 64  157.0     163.9 ( +4%)         421.4 (+168%)            422.1 (+169%)
```

## `tp.py`

Tensor parallelism's all-reduce traffic for Llama-3-70B on eight H100s.

In [4]:
!{sys.executable} tp.py

Llama-3-70B: 70.6B parameters, 141 GB in 16 bits; 17.6 GB per GPU, read in 5.3 ms
decode batch   all-reduce bytes per GPU per step   over NVLink   over PCIe Gen5   (weights: 5.3 ms)
       1                4.6 MB                      0.01 ms       0.07 ms
      16               73.4 MB                      0.16 ms       1.15 ms
      64              293.6 MB                      0.65 ms       4.59 ms
     256             1174.4 MB                      2.61 ms      18.35 ms


The post printed:

```text
Llama-3-70B: 70.6B parameters, 141 GB in 16 bits; 17.6 GB per GPU, read in 5.3 ms
decode batch   all-reduce bytes per GPU per step   over NVLink   over PCIe Gen5   (weights: 5.3 ms)
       1                4.6 MB                      0.01 ms       0.07 ms
      16               73.4 MB                      0.16 ms       1.15 ms
      64              293.6 MB                      0.65 ms       4.59 ms
     256             1174.4 MB                      2.61 ms      18.35 ms
```

## `steps.py`

Times decode steps (by batch size and cached tokens) and prompt prefills of Qwen2.5-1.5B, then fits the cost model the simulation uses. Writes `cost.npy` (included here).

**Note:** GPU; rewrites cost.npy for this machine (then rerun engine.py).

In [5]:
!{sys.executable} steps.py

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (181013 > 131072). Running this sequence through the model will result in indexing errors
decode, ms per step (tokens/s)
   B                C = 128             C = 512            C = 2048


   1         21.2 (   47/s)      21.6 (   46/s)      21.7 (   46/s)


   4         21.5 (  186/s)      22.1 (  181/s)      21.5 (  186/s)


  16         21.6 (  742/s)      22.1 (  724/s)      23.3 (  686/s)


  64         22.7 ( 2825/s)      23.3 ( 2749/s)                    


...

  T =  256:    22.6 ms (11352 tokens/s)


  T =  512:    30.6 ms (16758 tokens/s)


  T = 1024:    57.3 ms (17879 tokens/s)


  T = 2048:   108.8 ms (18821 tokens/s)


  T = 4096:   226.1 ms (18112 tokens/s)
fit: 21.0 ms + 18 us per new token + 24 ns per cached token read + 9121.5 ps x T^2; error over 21 steps: mean 4.7%, max 15.9%


The post printed:

```text
decode, ms per step (tokens/s)
   B                C = 128             C = 512            C = 2048
   1         46.0 (   22/s)      42.2 (   24/s)      55.5 (   18/s)
   4         49.9 (   80/s)      49.5 (   81/s)      66.1 (   61/s)
  16         74.0 (  216/s)      84.3 (  190/s)     131.5 (  122/s)
  64        127.7 (  501/s)     161.5 (  396/s)                    
 128        196.2 (  652/s)     280.3 (  457/s)                    
prefill, one sequence
  T =   16:    53.8 ms (  297 tokens/s)
  T =   64:    80.1 ms (  799 tokens/s)
  T =  128:   136.3 ms (  939 tokens/s)
  T =  256:   245.5 ms ( 1043 tokens/s)
  T =  512:   480.7 ms ( 1065 tokens/s)
  T = 1024:   990.9 ms ( 1033 tokens/s)
  T = 2048:  2082.3 ms (  984 tokens/s)
  T = 4096:  4303.0 ms (  952 tokens/s)
fit: 44.2 ms + 778 us per new token + 2334 ns per cached token read + 76514.7 ps x T^2; error over 21 steps: mean 6.7%, max 17.7%
```